# 📊 NDVI Delta Statistics Export

## Purpose

This notebook calculates annual vegetation change statistics from NDVI delta rasters.

The input rasters are expected to represent:

```text
NDVI delta = summer NDVI - spring NDVI
```

Negative values indicate vegetation decrease.

---

## Outputs

The notebook exports a CSV table containing:

- Mean NDVI delta
- Minimum and maximum NDVI delta
- Standard deviation
- Total valid area
- Vegetation loss area
- Vegetation loss percentage
- Area and percentage of 5 NDVI change severity classes

---

## Expected Input

Example raster filenames:

```text
NDVI_delta_2020.tif
NDVI_delta_2021.tif
NDVI_delta_2022.tif
```


## 1️⃣ Import Libraries

ArcPy is used for raster access, while NumPy is used for raster statistics.
The built-in `csv` module is used to export results.


In [7]:
import arcpy
import os
import csv
import numpy as np
from arcpy.sa import *

# Enable Spatial Analyst extension
arcpy.CheckOutExtension("Spatial")

# Allow overwriting outputs
arcpy.env.overwriteOutput = True

## 2️⃣ Define Input and Output Paths

Update these paths before running the notebook.


In [8]:
# Update these paths before running the notebook
input_folder = r"C:\Users\emese\Documents\ErasmusGIS\thesis\Data\vegetation\NDVI_delta_EOV"
output_csv = r"C:\Users\emese\Documents\ErasmusGIS\thesis\Data\vegetation\ndvi_delta_statistics2.csv"


## 3️⃣ Define NDVI Delta Severity Classes

These classes focus on vegetation loss.

Because the research question focuses on degradation, the negative NDVI delta range is divided into multiple loss classes.


In [9]:
# NDVI delta severity classes
# Delta = summer NDVI - spring NDVI

classes = [
    ("Very High Loss", -1.0, -0.6),
    ("High Loss", -0.6, -0.4),
    ("Moderate Loss", -0.4, -0.2),
    ("Low Loss", -0.2, 0.0)
]

## 4️⃣ Helper Function: Extract Year from Filename

The script expects the year to be present in the filename.


In [10]:
def extract_year(filename):
    """Extract the first 4-digit year from a filename."""
    digits = ''.join([c if c.isdigit() else ' ' for c in filename])
    parts = digits.split()

    for part in parts:
        if len(part) == 4:
            return int(part)

    return None

## 5️⃣ Calculate Statistics for Each NDVI Delta Raster

For each raster, the notebook calculates:

- Basic NDVI delta statistics
- Total valid area
- Area affected by vegetation decrease
- Area and percentage by severity class

Area is calculated from raster cell size.


In [11]:
results = []

# Read only clipped NDVI delta rasters

rasters = [
    f for f in os.listdir(input_folder)
    if f.lower().endswith((".tif", ".tiff"))
    and "clipped" in f.lower()
]

for f in sorted(rasters):

    year = extract_year(f)

    if year is None:
        print(f"Skipping file without detectable year: {f}")
        continue

    raster_path = os.path.join(input_folder, f)

    print(f"Processing {year}: {f}")

    raster = arcpy.Raster(raster_path)

    # Get cell size
    cell_width = float(arcpy.management.GetRasterProperties(raster, "CELLSIZEX").getOutput(0))
    cell_height = float(arcpy.management.GetRasterProperties(raster, "CELLSIZEY").getOutput(0))
    pixel_area_m2 = cell_width * cell_height
    pixel_area_km2 = pixel_area_m2 / 1_000_000

    # Convert raster to NumPy array
    arr = arcpy.RasterToNumPyArray(raster, nodata_to_value=np.nan)

    # Keep only valid pixels
    valid = arr[~np.isnan(arr)]

    if valid.size == 0:
        print(f"No valid pixels found in {f}")
        continue

    total_valid_pixels = valid.size
    total_area_km2 = total_valid_pixels * pixel_area_km2

    # Basic statistics
    mean_delta = float(np.mean(valid))
    min_delta = float(np.min(valid))
    max_delta = float(np.max(valid))
    std_delta = float(np.std(valid))

    # Vegetation loss: all negative values
    loss_pixels = np.sum(valid < 0)
    loss_area_km2 = loss_pixels * pixel_area_km2
    loss_percent = (loss_pixels / total_valid_pixels) * 100

    row = {
        "year": year,
        "raster": f,
        "mean_delta": mean_delta,
        "min_delta": min_delta,
        "max_delta": max_delta,
        "std_delta": std_delta,
        "total_area_km2": total_area_km2,
        "loss_area_km2": loss_area_km2,
        "loss_percent": loss_percent
    }

    # Severity class statistics
    for class_name, lower, upper in classes:

        class_pixels = np.sum((valid >= lower) & (valid < upper))
        class_area_km2 = class_pixels * pixel_area_km2
        class_percent = (class_pixels / total_valid_pixels) * 100

        safe_name = class_name.lower().replace(" ", "_").replace("/", "_")

        row[f"{safe_name}_area_km2"] = class_area_km2
        row[f"{safe_name}_percent"] = class_percent

    results.append(row)

print("\nStatistics calculation completed.")

Processing 2020: NDVI_delta_2020_clipped_EOV.tif
Processing 2021: NDVI_delta_2021_clipped_EOV.tif
Processing 2022: NDVI_delta_2022_clipped_EOV.tif
Processing 2023: NDVI_delta_2023_clipped_EOV.tif
Processing 2024: NDVI_delta_2024_clipped_EOV.tif
Processing 2025: NDVI_delta_2025_clipped_EOV.tif

Statistics calculation completed.


## 6️⃣ Export Results to CSV

The output CSV can be opened in Excel, ArcGIS Pro, QGIS, or used for charts.


In [12]:
if len(results) == 0:
    print("No results to export.")
else:
    fieldnames = list(results[0].keys())

    output_dir = os.path.dirname(output_csv)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)

    with open(output_csv, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)

    print("CSV exported:")
    print(output_csv)

CSV exported:
C:\Users\emese\Documents\ErasmusGIS\thesis\Data\vegetation\ndvi_delta_statistics2.csv


## 7️⃣ Optional Quick Preview

This cell prints the results table inside the notebook.


In [ ]:
for row in results:
    print(row)

## Interpretation

Suggested chart types based on the exported CSV:

### Mean NDVI Delta by Year

Use:

```text
x = year
y = mean_delta
```

This shows the average seasonal vegetation change per year.

---

### Vegetation Loss Area by Year

Use:

```text
x = year
y = loss_area_km2
```

This shows how much area experienced NDVI decrease.

---

### Severity Class Stacked Bar Chart

Use the class area columns:

```text
very_high_loss_area_km2
high_loss_area_km2
moderate_loss_area_km2
low_loss_area_km2
stable_or_gain_area_km2
```

This shows not only how much vegetation changed, but also how severe the change was.
